In [1]:
import ezdxf
import pandas as pd

doc = ezdxf.readfile(r"C:\Users\mvm\open3d_vision\data\25-GO1837-CAD-1 ind A_new.dxf")
msp = doc.modelspace()

rows = []

for e in msp:
    row = {
        "entity_type": e.dxftype(),
        "handle": e.dxf.handle
    }

    # Récupère TOUS les attributs DXF existants pour l'entité
    for key, value in e.dxfattribs().items():
        row[key] = value

    rows.append(row)

df = pd.DataFrame(rows)


In [2]:
import pandas as pd
import numpy as np

def drop_empty_columns(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.replace(
        to_replace=r"^\s*(NaN|nan|NAN)\s*$",
        value=np.nan,
        regex=True
    )
    return df_clean.dropna(axis=1, how="all")


drop_empty_columns(df)

,entity_type,handle,owner,layer,start,end,center,radius,start_angle,end_angle,...,ltscale,attribs_follow,name,xscale,yscale,zscale,true_color,degree,knot_tolerance,control_point_tolerance
0,LINE,14F8B,3F10F,LIMITE PARCELLAIRE,"(87923.333421, 101493.672646, 0.0)","(87928.224212, 101492.473508, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LINE,14F8D,3F10F,LIMITE PARCELLAIRE,"(87917.788227, 101442.491889, 0.0)","(87916.815417, 101442.133421, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LINE,14F8E,3F10F,LIMITE PARCELLAIRE,"(87930.580064, 101447.20559, 0.0)","(87917.788227, 101442.491889, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LINE,14F8F,3F10F,LIMITE PARCELLAIRE,"(87928.224212, 101492.473508, 0.0)","(87933.840401, 101491.096565, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LINE,14F90,3F10F,LIMITE PARCELLAIRE,"(87942.770384, 101488.889497, 0.0)","(87944.74138, 101488.402481, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8031,TEXT,3F108,3F10F,TEXTE-LOT-NUMERO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8032,TEXT,3F109,3F10F,TEXTE-LOT-NUMERO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8033,TEXT,3F10A,3F10F,TEXTE-LOT-NUMERO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8034,LINE,3F10C,3F10F,SURFACE-LOT,"(87927.552535, 101461.613223, 0.0)","(87927.258866, 101461.67454, 0.0)",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df["layer"].unique()

<ArrowStringArray>
['LIMITE PARCELLAIRE',             'PORTES',              'TEXTE',
          'TERRASSES',           'CLOISONS',        'MUR PORTEUR',
           'FENETRES',          'EXTERIEUR',            'PARKING',
        'SURFACE-LOT',          'ESCALIERS',          'TEXTE-LOT',
   'TEXTE-LOT-NUMERO',  'TEXTE-LOT-SURFACE',           'COTATION',
              'COUPE',      'COTATIONS MUR',    'CADRE-CARTOUCHE',
           'HACHURES']
Length: 19, dtype: str

Partie Classification des layers

In [4]:
"""
Classifieur Keras : chaîne de caractères -> layer DXF.
Entrée : une chaîne (ex. nom de layer, description).
Sortie : un des layers prédéfinis (EXTERIEUR, MUR PORTEUR, PORTES, etc.).
"""

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers


# Layers possibles (sorties du réseau), alignés avec le projet DXF
LAYER_LABELS = [
    "CLOISONS",
    "EXTERIEUR",
    "LIMITE PARCELLAIRE",
    "MUR PORTEUR",
    "PORTES",
    "ESCALIERS",
    "SURFACE-LOT",
    "TEXTE-LOT",
    "TEXTE-LOT-NUMERO",
    "OTHER",  # classe fourre-tout pour layers inconnus
]

# Paramètres du modèle
MAX_SEQ_LEN = 64
EMBED_DIM = 32
LSTM_UNITS = 64
DROPOUT = 0.3


def _build_char_vocab():
    """
    Construit le vocabulaire caractère : caractères imprimables + padding.
    Retourne (dict char->id, dict id->char).
    """
    # Caractères usuels (lettres, chiffres, espaces, tirets, etc.)
    chars = set(" 0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-.")
    # Ordre déterministe
    char_list = sorted(chars)
    # 0 = padding, 1 = inconnu (OOV)
    c2i = {"<PAD>": 0, "<OOV>": 1}
    for i, c in enumerate(char_list, start=2):
        c2i[c] = i
    i2c = {v: k for k, v in c2i.items()}
    return c2i, i2c


def encode_string(s, char2id, max_len=MAX_SEQ_LEN):
    """
    Encode une chaîne en séquence d'entiers (padding à max_len).
    Caractères inconnus -> id 1 (OOV).
    """
    if not isinstance(s, str):
        s = str(s)
    s = s.strip().upper()[:max_len]
    ids = [char2id.get(c, char2id["<OOV>"]) for c in s]
    # Padding à droite
    pad_len = max_len - len(ids)
    if pad_len > 0:
        ids += [char2id["<PAD>"]] * pad_len
    return np.array(ids, dtype=np.int32)


def decode_predictions(probs, labels=LAYER_LABELS):
    """
    Retourne la liste (label, proba) triée par proba décroissante.
    """
    idx = np.argsort(probs)[::-1]
    return [(labels[i], float(probs[i])) for i in idx]


def build_model(num_classes=None, max_len=MAX_SEQ_LEN, vocab_size=128, embed_dim=EMBED_DIM, lstm_units=LSTM_UNITS, dropout=DROPOUT):
    """
    Construit le réseau : Embedding -> LSTM -> Dense -> Softmax.
    num_classes : nombre de layers (défaut = len(LAYER_LABELS)).
    """
    if num_classes is None:
        num_classes = len(LAYER_LABELS)

    inputs = keras.Input(shape=(max_len,), dtype="int32")
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len)(inputs)
    x = layers.LSTM(lstm_units, dropout=dropout, recurrent_dropout=0.1)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def prepare_dataset(texts, labels, layer_list=None, char2id=None, max_len=MAX_SEQ_LEN):
    """
    Prépare X (séquences) et y (indices de classe) à partir de listes de chaînes et de noms de layer.
    labels : liste de noms de layer (doivent appartenir à layer_list).
    Retourne (X, y, char2id, layer_list).
    """
    if layer_list is None:
        layer_list = LAYER_LABELS
    if char2id is None:
        char2id, _ = _build_char_vocab()

    layer_to_id = {name: i for i, name in enumerate(layer_list)}
    # Classe "OTHER" pour tout label non présent dans layer_list
    other_id = layer_to_id.get("OTHER", len(layer_list))

    X = np.array([encode_string(t, char2id, max_len) for t in texts])
    y = np.array([layer_to_id.get(str(l).strip(), other_id) for l in labels], dtype=np.int32)
    return X, y, char2id, layer_list


def train_model(X, y, epochs=20, batch_size=32, validation_split=0.2, verbose=1):
    """
    Entraîne le modèle sur (X, y) et retourne l'historique et le modèle.
    """
    vocab_size = int(X.max()) + 1
    num_classes = len(np.unique(y))
    model = build_model(num_classes=num_classes, vocab_size=vocab_size)
    hist = model.fit(X, y, epochs=epochs, batch_size=batch_size, validation_split=validation_split, verbose=verbose)
    return model, hist


def predict_layer(model, string, char2id, layer_list=None):
    """
    Prédit le layer pour une chaîne.
    Retourne (label_prédit, proba, liste_triée (label, proba)).
    """
    if layer_list is None:
        layer_list = LAYER_LABELS
    seq = encode_string(string, char2id)
    seq_batch = np.expand_dims(seq, axis=0)
    probs = model.predict(seq_batch, verbose=0)[0]
    idx = int(np.argmax(probs))
    return layer_list[idx], float(probs[idx]), decode_predictions(probs, layer_list)


# --- Exemple d'utilisation avec données synthétiques ---
if __name__ == "__main__":
    # Données d'exemple : chaînes (variantes / bruit) -> layer
    texts = [
        # ================= CLOISONS =================
        "CLOISONS",
        "cloison",
        "Cloisons",
        "CLOISON",
        "Cloison_bureau_M",
        "cloison_chambre_1",
        "CLOISON_ETAGE_2",
        "cloison_sdb",
        "CLOISONS_RDC",
        "cloison-placo",
        "cloisons_int",
        "CLOISON_INTERIEURE",
        "cloison bureau nord",
        "CLOISON_BUREAU_SUD",
        "clois_bureau",
        "cloison_local_tech",
        "CLOISON_01",
        "cloison-A",
        "CLN_BUREAU",

        # ================= MUR PORTEUR =================
        "MUR PORTEUR",
        "mur porteur",
        "MUR PORTEUR ",
        "Mur_Porteur_RDC",
        "mur_porteur_etage",
        "MUR_PORTEUR_BETON",
        "mur_beton_20cm",
        "MUR_STRUCTURE",
        "structure_mur",
        "mur_principal",
        "MUR_EXT_PORTEUR",
        "mur_porteur_axe_A",
        "MP_RDC",
        "mur-structurel",
        "mur_refend",
        "MUR_REFEND_ETAGE",
        "voile_beton",
        "VOILE_BETON_RDC",

        # ================= PORTES =================
        "PORTES",
        "porte",
        "PORTE",
        "Portes",
        "porte_chambre_2",
        "PORTE_ENTREE",
        "porte_sdb",
        "portes_RDC",
        "PORTE-01",
        "porte_bureau_nord",
        "ouvrant_porte",
        "porte_int",
        "porte_ext",
        "PORTE_ETAGE_1",
        "door_room_1",
        "porte_coupe_feu",
        "porte_CF",
        "P_RDC",

        # ================= EXTERIEUR =================
        "EXTERIEUR",
        "exterieur",
        "EXTERIEUR ",
        "facade",
        "FACADE_NORD",
        "facade_sud",
        "mur_ext",
        "MUR_EXTERIEUR",
        "contour_batiment",
        "BATIMENT_EXT",
        "enveloppe",
        "ENVELOPPE_THERMIQUE",
        "toiture_ext",
        "TOITURE",
        "terrain",
        "TERRAIN_NATUREL",
        "limite_parcelle",

        # ================= ESCALIERS =================
        "ESCALIERS",
        "escalier",
        "ESCALIER",
        "Escalier_RDC",
        "escalier_principal",
        "ESCALIER_SECOURS",
        "escalier_etage_2",
        "ESC_RDC",
        "escaliers_int",
        "vollee_escalier",
        "ESCALIER_METAL",
        "escalier_beton",
        "ESC_BETON",
        "escalier_exterieur",
    ]

    labels = [
        # CLOISONS (19)
        *["CLOISONS"] * 19,

        # MUR PORTEUR (18)
        *["MUR PORTEUR"] * 18,

        # PORTES (18)
        *["PORTES"] * 18,

        # EXTERIEUR (17)
        *["EXTERIEUR"] * 17,

        # ESCALIERS (14)
        *["ESCALIERS"] * 14,
    ]


    X, y, char2id, layer_list = prepare_dataset(texts, labels)
    vocab_size = max(int(X.max()) + 1, len(char2id))
    model = build_model(num_classes=len(layer_list), vocab_size=vocab_size)
    model.fit(X, y, epochs=2, batch_size=8, verbose=1)

    # Test
    for s in ["CLOISONS", "mur porteur", "inconnu_layer"]:
        label, prob, ranked = predict_layer(model, s, char2id, layer_list)
        print(f"  '{s}' -> {label} ({prob:.2f})")


Epoch 1/2


c:\Users\mvm\open3d_vision\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1860 - loss: 2.2585 
Epoch 2/2
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2674 - loss: 2.0341
  'CLOISONS' -> MUR PORTEUR (0.30)
  'mur porteur' -> MUR PORTEUR (0.30)
  'inconnu_layer' -> MUR PORTEUR (0.30)


 RESET

In [5]:
import shapely.geometry as geom
import shapely.ops as ops
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [6]:
WALL_HEIGHT = 3.0       # hauteur murs
WALL_THICKNESS = 0.5   # épaisseur murs
SLAB_THICKNESS = 0.3   # dalle
COLUMN_HEIGHT = 3.0
ARC_RESOLUTION = 32       # nombre de segments par arc
ELLIPSE_RESOLUTION = 48   # nombre de segments pour une ellipse complète
SPLINE_RESOLUTION = 64
LAYER_COLORS = {
    "EXTERIEUR": [0.8, 0.2, 0.2],
    "MUR PORTEUR": [0.7, 0.7, 0.7],
    "PORTES": [0.2, 0.6, 0.2],
    "ESCALIERS": [0.3, 0.5, 0.8],
    "CLOISONS": [0.9, 0.7, 0.4],
    "LIMITE PARCELLAIRE": [0.4, 0.4, 0.6],
}
DEFAULT_COLOR = [0.3, 0.5, 0.8]



In [7]:

def color_from_layer(layer):
    """Retourne la couleur RGB [0-1] associée au layer, ou DEFAULT_COLOR si inconnu."""
    return LAYER_COLORS.get(layer, DEFAULT_COLOR)


In [8]:
def enrich_spline_geometry(df, msp):
    geom_map = {}

    for e in msp.query("SPLINE"):
        pts = spline_to_polyline(e, SPLINE_RESOLUTION)
        if e.closed and pts:
            pts.append(pts[0])
        geom_map[e.dxf.handle] = pts

    df["geometry"] = df["geometry"].where(
        df["entity_type"] != "SPLINE",
        df["handle"].map(geom_map)
    )

    return df


In [9]:
def spline_to_polyline(spline_entity, resolution=SPLINE_RESOLUTION):
    """
    Discrétise une SPLINE DXF en polyligne 3D
    """
    try:
        bspline = spline_entity.construction_tool()
        points = [
            tuple(bspline.point(t))
            for t in np.linspace(0, 1, resolution)
        ]
        return [(x, y, z) for x, y, z in points]
    except Exception:
        return []


In [10]:
def enrich_lwpolyline_geometry(df, msp):
    """
    Ajoute la colonne 'geometry' pour les LWPOLYLINE
    """
    geom_map = {}

    for e in msp.query("LWPOLYLINE"):
        points = [(x, y, 0.0) for x, y, *_ in e]

        if e.closed and points:
            points.append(points[0])

        geom_map[e.dxf.handle] = points

    # Créer la colonne geometry si elle n'existe pas (KeyError sinon)
    if "geometry" not in df.columns:
        df["geometry"] = np.nan
    df["geometry"] = df["geometry"].where(
        df["entity_type"] != "LWPOLYLINE",
        df["handle"].map(geom_map)
    )

    return df


df = enrich_lwpolyline_geometry(df, msp)
df = enrich_spline_geometry(df,msp)

In [11]:
import open3d as o3d
import numpy as np

def wall_from_line(start, end, height, thickness,layer = None, eps = 1e-6):
    x1, y1, _ = start
    x2, y2, _ = end

    x1, x2 = -x1, -x2

    direction = np.array([x2 - x1, y2 - y1], dtype = float)
    length = np.linalg.norm(direction)
    if length < eps:
        direction = np.array([1.0, 0.0])   # direction par défaut (axe X)
        length = 1.0
    else:
        direction /= length

        
    normal = np.array([-direction[1], direction[0]])

    p1 = np.array([x1, y1]) + normal * thickness / 2
    p2 = np.array([x2, y2]) + normal * thickness / 2
    p3 = np.array([x2, y2]) - normal * thickness / 2
    p4 = np.array([x1, y1]) - normal * thickness / 2

    vertices = np.array([
        [*p1, 0], [*p2, 0], [*p3, 0], [*p4, 0],
        [*p1, height], [*p2, height], [*p3, height], [*p4, height]
    ])

#variable représentant les sommets du quadrilatère du mur
#    7 -------- 6
#   /|         /|
#  4 -------- 5 |
#  | |        | |
#  | 3 ------ | 2
#  |/         |/
#  0 -------- 1

    triangles = np.array([
        [0,1,2],[0,2,3],        # bottom
        [4,5,6],[4,6,7],        # top
        [0,1,5],[0,5,4],
        [1,2,6],[1,6,5],
        [2,3,7],[2,7,6],
        [3,0,4],[3,4,7]
    ])

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.triangles = o3d.utility.Vector3iVector(triangles)


# indices des triangles latéraux uniquement
    wall_dir = np.array([x2 - x1, y2 - y1, 0])
    wall_dir /= np.linalg.norm(wall_dir)
    wall_center = vertices.mean(axis=0)
    mesh.compute_triangle_normals()

    tri_normals = np.asarray(mesh.triangle_normals)
    triangles = np.asarray(mesh.triangles)
    eps = 1e-6
    lateral_mask = np.abs(tri_normals[:, 2]) < eps

    for i in np.where(lateral_mask)[0]:
        tri = triangles[i]
        face_center = vertices[tri].mean(axis=0)

        outward_dir = face_center - wall_center
        if np.dot(tri_normals[i], outward_dir) < 0:
            triangles[i] = triangles[i][::-1]

    mesh.triangles = o3d.utility.Vector3iVector(triangles)
    mesh.compute_vertex_normals()
    mesh.paint_uniform_color(color_from_layer(layer))
    return mesh


In [12]:
def normalize_points(points):
    """
    Convertit tous les points en tuples (x, y, z)
    Filtre les points invalides
    """
    clean = []
    for p in points:
        if p is None:
            continue

        # ezdxf Vec3
        if hasattr(p, "x"):
            clean.append((p.x, p.y, p.z))
            continue

        # tuple / list
        if isinstance(p, (tuple, list)) and len(p) >= 2:
            z = p[2] if len(p) > 2 else 0.0
            clean.append((p[0], p[1], z))

    return clean


In [13]:
def extrude_mesh(mesh_2d, height):
    """
    Extrude un TriangleMesh plan (z=0) vers +z
    """
    vertices = np.asarray(mesh_2d.vertices)
    triangles = np.asarray(mesh_2d.triangles)

    n = len(vertices)

    # sommets du dessus
    top_vertices = vertices.copy()
    top_vertices[:, 2] += height

    all_vertices = np.vstack([vertices, top_vertices])

    faces = []

    # faces du bas
    faces.extend(triangles.tolist())

    # faces du haut (sens inverse)
    faces.extend((triangles[:, ::-1] + n).tolist())

    # faces latérales
    edges = set()
    for tri in triangles:
        edges.add(tuple(sorted((tri[0], tri[1]))))
        edges.add(tuple(sorted((tri[1], tri[2]))))
        edges.add(tuple(sorted((tri[2], tri[0]))))

    for i, j in edges:
        faces.append([i, j, j + n])
        faces.append([i, j + n, i + n])

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(all_vertices)
    mesh.triangles = o3d.utility.Vector3iVector(np.array(faces))
    mesh.compute_vertex_normals()

    return mesh


In [14]:
def wall_from_polyline(points, height, thickness, layer = None):
    meshes = []

    pts = normalize_points(points)

    if len(pts) < 2:
        return meshes

    # évite le segment nul si polyligne fermée
    for p1, p2 in zip(pts[:-1], pts[1:]):
        if p1[:2] == p2[:2]:
            continue

        mesh = wall_from_line(p1, p2, height, thickness, layer = layer)
        if mesh is not None:
            meshes.append(mesh)

    return meshes



In [15]:
def column_from_circle(center, radius, height, layer=None):
    """Crée un cylindre (poteau) centré, coloré selon le layer."""
    cx, cy, _ = center
    mesh = o3d.geometry.TriangleMesh.create_cylinder(
        radius=radius,
        height=height
    )
    mesh.translate((cx, cy, height / 2))
    mesh.compute_vertex_normals()
    mesh.paint_uniform_color(color_from_layer(layer))
    return mesh


In [16]:
def slab_from_polygon_2d(poly2d, thickness, layer = None):
    if len(poly2d) < 3:
        return None

    poly2d = [(-x, y) for x, y in poly2d]

    if poly2d[0][0] != poly2d[-1][0] or poly2d[0][1] != poly2d[-1][1]:
        poly2d = poly2d + [poly2d[0]]

    polygon = geom.Polygon(poly2d)
    if not polygon.is_valid:
        return None

    triangles = ops.triangulate(polygon)

    vertices = []
    faces = []

    for tri in triangles:
        coords = list(tri.exterior.coords)[:-1]
        idx = []
        for x, y in coords:
            vertices.append([x, y, 0.0])
            idx.append(len(vertices) - 1)
        faces.append(idx)

    mesh_2d = o3d.geometry.TriangleMesh()
    mesh_2d.vertices = o3d.utility.Vector3dVector(np.array(vertices))
    mesh_2d.triangles = o3d.utility.Vector3iVector(np.array(faces))
    mesh_3d = extrude_mesh(mesh_2d, thickness)
    mesh_3d.paint_uniform_color(color_from_layer(layer))
    return mesh_3d


In [17]:
def slab_from_lwpolyline(points, thickness, layer=None):
    """
    points : liste de tuples (x, y [, z]). Couleur selon layer.
    """
    poly2d = np.array([[p[0], p[1]] for p in points])
    slab = slab_from_polygon_2d(poly2d, thickness, layer=layer)
    if slab is not None:
        slab.compute_vertex_normals()
    return slab


In [18]:
import math
import numpy as np

def arc_to_polyline(center, radius, start_angle, end_angle, resolution):
    """
    Angles en degrés DXF
    Retourne une liste de points (x, y, 0)
    """
    cx, cy, _ = center

    angles = np.linspace(
        math.radians(start_angle),
        math.radians(end_angle),
        resolution
    )

    return [
        (
            cx + radius * math.cos(a),
            cy + radius * math.sin(a),
            0.0
        )
        for a in angles
    ]


In [19]:
def resolve_geometry_block(doc, block_name):
    """
    Retourne les entités contenues dans un block anonyme (*Dxxx)
    """
    if not isinstance(block_name, str):
        return []

    if not block_name.startswith("*"):
        return []

    try:
        block = doc.blocks[block_name]
    except KeyError:
        return []

    return list(block)


In [20]:
def extract_points_from_block(block):
    """
    Extrait les sommets 2D contenus dans un block
    """
    for e in block:
        if e.dxftype() == "LWPOLYLINE":
            return [(x, y, 0.0) for x, y, *_ in e]
    return None


In [21]:
def lwpolyline_to_points(row):
    """
    Extrait les sommets d'une LWPOLYLINE depuis le DataFrame.
    Retourne une liste de points (x, y, z)
    """
    if row["geometry"] is None:
        return []

    points = []

    for p in row["geometry"]:
        if hasattr(p, "x"):
            points.append((p.x, p.y, p.z))
        elif isinstance(p, (tuple, list)):
            z = p[2] if len(p) > 2 else 0.0
            points.append((p[0], p[1], z))

    return points


In [22]:
def slab_from_lwpolyline(points, thickness, layer=None):
    """Dalle à partir d'une polyligne fermée, colorée selon le layer."""
    poly = normalize_points(points)
    if len(poly) < 3:
        return None
    if poly[0][:2][0] != poly[-1][:2][0] or poly[0][:2][1] != poly[-1][:2][1]:
        poly.append(poly[0])
    vertices = np.array([[x, y, 0] for x, y, _ in poly])
    faces = [[i, i + 1, 0] for i in range(1, len(vertices) - 1)]
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.triangles = o3d.utility.Vector3iVector(faces)
    slab = extrude_mesh(mesh, thickness)
    slab.compute_vertex_normals()
    slab.paint_uniform_color(color_from_layer(layer))
    return slab


In [23]:
def walls_from_lwpolyline(points, height, thickness, layer = None):
    pts = normalize_points(points)

    if len(pts) < 2:
        return []

    return wall_from_polyline(pts, height, thickness, layer = layer)


In [24]:
def ellipse_to_polyline(center, major_axis, ratio, start_param, end_param, resolution):
    """
    Discrétise une ellipse DXF en polyligne 3D.
    center: (x, y, z), major_axis: vecteur demi-grand axe, ratio: petit/grand axe.
    start_param, end_param: en radians (0 à 2*pi).
    Retourne une liste de points (x, y, z).
    """
    cx, cy, cz = _point_xyz(center)
    mx, my, mz = _vec_xyz(major_axis)
    length = np.sqrt(mx * mx + my * my + mz * mz)
    if length < 1e-10:
        return []
    # Demi-grand axe (DXF: major_axis = du centre vers une extrémité)
    ax, ay = mx / length, my / length
    # Demi-petit axe perpendiculaire dans le plan XY, longueur = ratio * length
    bx = -ratio * ay
    by = ratio * ax
    t_vals = np.linspace(float(start_param), float(end_param), max(2, resolution))
    return [
        (cx + length * (np.cos(t) * ax + np.sin(t) * bx),
         cy + length * (np.cos(t) * ay + np.sin(t) * by),
         cz)
        for t in t_vals
    ]


def _point_xyz(p):
    """Retourne (x, y, z) depuis un point DXF (tuple ou Vec3)."""
    if hasattr(p, "x"):
        return (p.x, p.y, p.z)
    return (p[0], p[1], p[2] if len(p) > 2 else 0.0)


def _vec_xyz(v):
    """Retourne (x, y, z) depuis un vecteur DXF."""
    if hasattr(v, "x"):
        return (v.x, v.y, v.z)
    return (v[0], v[1], v[2] if len(v) > 2 else 0.0)

In [25]:
mirror_y = np.array([
    [-1,  0,  0,  0],
    [ 0,  1,  0,  0],
    [ 0,  0,  1,  0],
    [ 0,  0,  0,  1]
])


In [26]:
import numpy as np

def filter_outliers_df(df, coord_columns=("start", "end", "center"), zscore_threshold=4.0):
    """
    Retire les outliers (éloignés sur X ou Y) du DataFrame basé sur une valeur z-score.
    S'applique sur les colonnes spécifiées contenant des coordonnées ou points 3D.
    """
    coords = []
    idx_map = []

    # Collecte tous les points contenus dans les colonnes de coordonnées (start, end, center, etc.)
    for i, row in df.iterrows():
        for col in coord_columns:
            val = row.get(col)
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                try:
                    x, y, *_ = _point_xyz(val)
                    coords.append([x, y])
                    idx_map.append(i)
                except Exception:
                    continue

    if not coords:
        return df  # rien à filtrer

    coords = np.array(coords)
    mean = coords.mean(axis=0)
    std = coords.std(axis=0)
    zscores = np.abs((coords - mean) / (std + 1e-9))  # éviter division par 0

    ok = (zscores < zscore_threshold).all(axis=1)
    ok_idxs = set(i for j, i in enumerate(idx_map) if ok[j])

    df_filtered = df[df.index.isin(ok_idxs)].reset_index(drop=True)
    # On retourne le même DataFrame si rien n'est filtré
    return df_filtered if len(df_filtered) < len(df) else df

def dataframe_to_point_cloud_with_lines(df, color_by_layer=True, filter_outliers=True):
    """
    Extrait tous les points 3D du dataframe DXF et les retourne en nuage de points Open3D.
    Si certains points sont connectés (ex: les extrémités des lignes), ajoute aussi les points intermédiaires
    représentant les lignes qui les relient.
    Les couleurs sont optionnellement définies par le layer (LAYER_COLORS).
    Optionnellement filtre les outliers dans le dataframe avant extraction.
    """
    if filter_outliers:
        df = filter_outliers_df(df)

    points_list = []
    colors_list = []

    # Nouvelle liste pour stocker les lignes comme séquences de points
    line_points = []
    line_colors = []

    for _, row in df.iterrows():
        layer = row.get("layer")
        rgb = np.array(color_from_layer(layer)) if color_by_layer else np.array(DEFAULT_COLOR)
        etype = row.get("entity_type")

        # LINE : points de début/fin + ajout de la ligne en points interpolés
        if etype == "LINE":
            start = row.get("start")
            end = row.get("end")
            if (
                start is not None
                and end is not None
                and not (isinstance(start, float) and np.isnan(start))
                and not (isinstance(end, float) and np.isnan(end))
            ):
                xs, ys, zs = _point_xyz(start)
                xe, ye, ze = _point_xyz(end)
                # Ajout des extrémités dans le nuage de points principal
                points_list.append([-xs, ys, zs])
                colors_list.append(rgb)
                points_list.append([-xe, ye, ze])
                colors_list.append(rgb)

                # Discrétise la ligne entre start et end, et ajoute comme points
                n_line_pts = max(2, int(np.linalg.norm(np.array([xs - xe, ys - ye, zs - ze])) * 5))
                t_vals = np.linspace(0, 1, n_line_pts)
                for t in t_vals:
                    px = xs * (1 - t) + xe * t
                    py = ys * (1 - t) + ye * t
                    pz = zs * (1 - t) + ze * t
                    line_points.append([-px, py, pz])
                    line_colors.append(rgb)

        # ARC : discrétisation
        elif etype == "ARC":
            c, r = row.get("center"), row.get("radius")
            sa, ea = row.get("start_angle"), row.get("end_angle")
            if c is not None and r is not None and sa is not None and ea is not None and not (np.isnan(r) or np.isnan(sa) or np.isnan(ea)):
                pts = arc_to_polyline(c, r, sa, ea, ARC_RESOLUTION)
                for p in pts:
                    x, y, z = p[0], p[1], p[2]
                    points_list.append([-x, y, z])
                    colors_list.append(rgb)

        # LWPOLYLINE / SPLINE : geometry
        elif etype in ("LWPOLYLINE", "SPLINE"):
            geom = row.get("geometry")
            if geom is None or not isinstance(geom, (list, tuple)):
                continue
            poly_pts = []
            for p in geom:
                if p is None:
                    continue
                x, y, z = _point_xyz(p)
                poly_pts.append([-x, y, z])
                points_list.append([-x, y, z])
                colors_list.append(rgb)
            # relie les points successifs de la polyligne par segments (en points)
            if len(poly_pts) > 1:
                for i in range(len(poly_pts) - 1):
                    x1, y1, z1 = poly_pts[i]
                    x2, y2, z2 = poly_pts[i + 1]
                    n_line_pts = max(2, int(np.linalg.norm(np.array([x1 - x2, y1 - y2, z1 - z2])) * 5))
                    t_vals = np.linspace(0, 1, n_line_pts)
                    for t in t_vals:
                        px = x1 * (1 - t) + x2 * t
                        py = y1 * (1 - t) + y2 * t
                        pz = z1 * (1 - t) + z2 * t
                        line_points.append([px, py, pz])
                        line_colors.append(rgb)

        # CIRCLE : centre (+ optionnellement points sur le cercle)
        elif etype == "CIRCLE":
            c = row.get("center")
            r = row.get("radius")
            if c is not None:
                x, y, z = _point_xyz(c)
                points_list.append([-x, y, z])
                colors_list.append(rgb)
            if c is not None and r is not None and not np.isnan(r):
                pts = arc_to_polyline(c, r, 0, 360, ARC_RESOLUTION)
                for p in pts:
                    points_list.append([-p[0], p[1], p[2]])
                    colors_list.append(rgb)

        # ELLIPSE : discrétisation
        elif etype == "ELLIPSE":
            geom = row.get("geometry")
            if geom is not None and isinstance(geom, (list, tuple)):
                for p in geom:
                    if p is not None:
                        x, y, z = _point_xyz(p)
                        points_list.append([-x, y, z])
                        colors_list.append(rgb)
            else:
                center = row.get("center")
                if center is not None:
                    pts = ellipse_to_polyline(
                        center, row.get("major_axis"), row.get("ratio"),
                        row.get("start_param"), row.get("end_param"),
                        ELLIPSE_RESOLUTION
                    )
                    for p in pts:
                        points_list.append([-p[0], p[1], p[2]])
                        colors_list.append(rgb)

    # Ajoute les points des lignes connectées au nuage de points global
    # (cela permet de "dessiner" les lignes comme lignes de points)
    all_points = np.concatenate([points_list, line_points], axis=0) if line_points else np.array(points_list)
    all_colors = np.concatenate([colors_list, line_colors], axis=0) if line_colors else np.array(colors_list)

    if all_points.shape[0] == 0:
        return None

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.array(all_points))
    pcd.colors = o3d.utility.Vector3dVector(np.array(all_colors))
    return pcd

# Utilise la nouvelle fonction
pcd = dataframe_to_point_cloud_with_lines(df, color_by_layer=True, filter_outliers=True)
if pcd is not None:
    print(f"Nuage de points et segments : {len(np.asarray(pcd.points))} points")
    o3d.visualization.draw_geometries([pcd], point_show_normal=False)
else:
    print("Aucun point extrait du dataframe.")

Nuage de points et segments : 59806 points


In [27]:
o3d.io.write_point_cloud("copy_of_fragment.pcd", pcd)

True

In [28]:
meshes = []

for _, row in df.iterrows():

    if row["entity_type"] == "LINE" and row["start"] and row["end"]:
        
        wall = wall_from_line(
            row["start"],
            row["end"],
            WALL_HEIGHT,
            WALL_THICKNESS,
            layer = row["layer"]
        )
        if wall:
            
            meshes.append(wall)
    
    if row["entity_type"] == "HATCH" and pd.notna(row["geometry"]):

        block_entities = resolve_geometry_block(doc, row["geometry"])
        points = extract_points_from_block(block_entities)

        if points and len(points) >= 3:
            slab = slab_from_lwpolyline(points, SLAB_THICKNESS, layer=row["layer"])
            meshes.append(slab)

    elif row["entity_type"] == "CIRCLE" and row["center"] and row["radius"]:
        col = column_from_circle(
            row["center"],
            row["radius"],
            COLUMN_HEIGHT,
            layer=row["layer"]
        )
        meshes.append(col)


    elif row["entity_type"] == "ARC":
        
        points = arc_to_polyline(
        row["center"],
        row["radius"],
        row["start_angle"],
        row["end_angle"],
        ARC_RESOLUTION
    )
        meshes.extend(
            wall_from_polyline(
                points,
                WALL_HEIGHT,
                WALL_THICKNESS,
                layer=row["layer"]
            )
        )
    elif row["entity_type"] == "SPLINE":

        points = row["geometry"]

        if not points or len(points) < 2:
            continue

        # spline fermée → dalle
        if points[0][:2] == points[-1][:2]:
            slab = slab_from_polygon_2d(
                [(x, y) for x, y, _ in points],
                SLAB_THICKNESS,
                layer=row["layer"]
            )
            if slab:
                meshes.append(slab)

        # spline ouverte → mur courbe
        else:
            meshes.extend(
                wall_from_polyline(
                    points,
                    WALL_HEIGHT,
                    WALL_THICKNESS,
                    layer=row["layer"]
                )
            )

    # #GESTION DES ELLIPSES, A AJOUTER SEULEMENT SI PERTINENT
    # elif row["entity_type"] == "ELLIPSE":

    #     points = ellipse_to_polyline(
    #         row["center"],
    #         row["major_axis"],
    #         row["ratio"],
    #         row["start_param"],
    #         row["end_param"],
    #         ELLIPSE_RESOLUTION
    #     )

    #     if len(points) < 2:
    #         continue

    #     # ellipse fermée → dalle
    #     if abs(row["end_param"] - row["start_param"]) >= 2 * np.pi - 1e-3:
    #         slab = slab_from_lwpolyline(
    #             points + [points[0]],
    #             SLAB_THICKNESS
    #         )
    #         if slab:
    #             meshes.append(slab)

    #     # ellipse ouverte → mur courbe
    #     else:
    #         meshes.extend(
    #             wall_from_polyline(
    #                 points,
    #                 WALL_HEIGHT,
    #                 WALL_THICKNESS
    #             )
    #         )

#   #  GESTION DE LA POLYLINE, A AJOUTER SEULEMENT SI PERTINENT
    # elif row["entity_type"] == "LWPOLYLINE":

    #     points = lwpolyline_to_points(row)

    #     if not points:
    #         continue

    #     # polyligne fermée → dalle
    #     if points[0][:2] == points[-1][:2]:
    #         slab = slab_from_lwpolyline(
    #             points,
    #             SLAB_THICKNESS
    #         )
    #         if slab:
    #             meshes.append(slab)

    #     # polyligne ouverte → murs
    #     else:
    #         thickness = (
    #             row["const_width"]
    #             if row.get("const_width") not in (None, 0, -1)
    #             else WALL_THICKNESS
    #         )

    #         meshes.extend(
    #             walls_from_lwpolyline(
    #                 points,
    #                 WALL_HEIGHT,
    #                 thickness
    #             )
    #         )


C:\Users\mvm\AppData\Local\Temp\ipykernel_23912\1276610980.py:56: RuntimeWarning: invalid value encountered in divide
  wall_dir /= np.linalg.norm(wall_dir)
C:\Users\mvm\AppData\Local\Temp\ipykernel_23912\1276610980.py:56: RuntimeWarning: invalid value encountered in divide
  wall_dir /= np.linalg.norm(wall_dir)
C:\Users\mvm\AppData\Local\Temp\ipykernel_23912\1276610980.py:56: RuntimeWarning: invalid value encountered in divide
  wall_dir /= np.linalg.norm(wall_dir)
C:\Users\mvm\AppData\Local\Temp\ipykernel_23912\1276610980.py:56: RuntimeWarning: invalid value encountered in divide
  wall_dir /= np.linalg.norm(wall_dir)


In [29]:
scene = o3d.geometry.TriangleMesh()
for i in range(min(6000, len(meshes))):
    scene += meshes[i]

print(f"Scene : {len(scene.vertices)} sommets, {len(scene.triangles)} triangles")
scene.compute_vertex_normals()

# Centrer à l'origine : coordonnées DXF (60k+) hors du champ de la caméra par défaut
center = scene.get_center()
scene.translate(-center)

# Option 1 : draw() (API moderne, souvent plus fiable en Jupyter/WebRTC)
#o3d.visualization.draw([scene], title="Scène DXF")

# Option 2 : si vide, essayer draw_geometries avec mesh_show_back_face (normales inversées)
o3d.visualization.draw_geometries([scene], mesh_show_back_face=True)

# Option 3 : fenêtre OpenGL native (dans un terminal, après l'export PLY) :
#   cd c:\Users\mvm\open3d_vision\src
#   python run_visualization.py


Scene : 48000 sommets, 72000 triangles


Export PLY

In [30]:
ply_filename = "2emeetage.ply"
o3d.io.write_triangle_mesh(ply_filename, scene)

True

In [31]:
# Lancer la lecture du PLY exporté (fenêtre OpenGL native)
import subprocess
import sys
from pathlib import Path

run_script = Path.cwd() / "run_visualization.py"
subprocess.run([sys.executable, str(run_script), ply_filename], cwd=Path.cwd())

CompletedProcess(args=['c:\\Users\\mvm\\open3d_vision\\.venv\\Scripts\\python.exe', 'c:\\Users\\mvm\\open3d_vision\\src\\run_visualization.py', '2emeetage.ply'], returncode=0)

In [32]:
df["layer"][500:550]

500    CLOISONS
501    CLOISONS
502    CLOISONS
503    CLOISONS
504    CLOISONS
505    CLOISONS
506    CLOISONS
507    CLOISONS
508    CLOISONS
509    CLOISONS
510    CLOISONS
511    CLOISONS
512    CLOISONS
513    CLOISONS
514    CLOISONS
515    CLOISONS
516    CLOISONS
517    CLOISONS
518      PORTES
519      PORTES
520      PORTES
521      PORTES
522      PORTES
523      PORTES
524      PORTES
525      PORTES
526      PORTES
527      PORTES
528      PORTES
529      PORTES
530      PORTES
531      PORTES
532      PORTES
533      PORTES
534      PORTES
535      PORTES
536      PORTES
537      PORTES
538      PORTES
539      PORTES
540      PORTES
541      PORTES
542      PORTES
543      PORTES
544      PORTES
545      PORTES
546      PORTES
547      PORTES
548      PORTES
549      PORTES
Name: layer, dtype: str

In [33]:
import numpy as np


def classify_entity(row,wall_min_length=0.5,wall_max_length=50,door_arc_max_radius=2.0,column_max_radius=0.5):
    """
    Heuristic classification of a DXF entity
    independent from layer naming.
    """

    etype = row.get("entity_type")

    # --------------------------------------------------
    # 1️⃣ TEXT / MTEXT → annotation
    # --------------------------------------------------
    if etype in ["TEXT", "MTEXT"]:
        return "annotation"

    # --------------------------------------------------
    # 2️⃣ CIRCLE → column or small object
    # --------------------------------------------------
    if etype == "CIRCLE":
        # Récupère le rayon du cercle DXF
        radius = row.get("radius")
        # Vérifie que le rayon existe et est valide (non NaN)
        if radius is not None and not np.isnan(radius):
            # Cercle de petit rayon → interprété comme poteau (colonne)
            if radius < column_max_radius:
                return "column"
        # Rayon absent, invalide ou trop grand → type inconnu
        return "unknown"

    # --------------------------------------------------
    # 3️⃣ ARC → probable door (arc attached to wall)
    # --------------------------------------------------
    if etype == "ARC":
        radius = row.get("radius")
        if radius is not None and not np.isnan(radius):
            if radius < door_arc_max_radius:
                return "door"
        return "unknown"

    # --------------------------------------------------
    # 4️⃣ LINE → possible wall
    # --------------------------------------------------
    if etype == "LINE":
        start = row.get("start")
        end = row.get("end")

        if start is not None and end is not None:
            length = np.linalg.norm(np.array(end) - np.array(start))

            if wall_min_length < length < wall_max_length:
                return "wall"

        return "unknown"

    # --------------------------------------------------
    # 5️⃣ LWPOLYLINE
    # --------------------------------------------------
    if etype == "LWPOLYLINE":

        geometry = row.get("geometry")

        if geometry and isinstance(geometry, list):

            # fermé → slab / pièce
            if geometry[0] == geometry[-1]:
                return "slab"

            # ouvert → mur segmenté
            if len(geometry) >= 2:
                return "wall"

        return "unknown"

    # --------------------------------------------------
    # fallback
    # --------------------------------------------------
    return "unknown"


In [34]:
# Export SVG : source = DXF (précis, vecteurs natifs) ou fallback = nuage de points (projection XY)
svg_filename = ply_filename.replace(".ply", ".svg") if "ply_filename" in dir() else "scene.svg"

    # DXF → SVG : le plus précis (lignes, polylignes, arcs natifs conservés)
from ezdxf.addons.drawing import Frontend, RenderContext, svg, layout, config
context = RenderContext(doc)
backend = svg.SVGBackend()
cfg = config.Configuration(background_policy=config.BackgroundPolicy.WHITE, color_policy=config.ColorPolicy.BLACK)
frontend = Frontend(context, backend, config=cfg)
frontend.draw_layout(msp)
page = layout.Page(0, 0, layout.Units.mm, margins=layout.Margins.all(2))
svg_content = backend.get_string(page)
with open(svg_filename, "wt", encoding="utf8") as f:
    f.write(svg_content)
print(f"Export SVG (source DXF) : {svg_filename}")


Export SVG (source DXF) : 2emeetage.svg
